In [4]:
# Cell 1
# Mục đích: thiết lập kết nối và hàm đo byte, dùng lại cho cả ngày.
import os
from pathlib import Path
from dotenv import load_dotenv
from google.cloud import bigquery

load_dotenv()                                  # đọc file .env vào biến môi trường
OLIST_DIR = Path(os.getenv("OLIST_DIR", ""))   # thư mục chứa 9 file CSV
PROJECT = os.getenv("GCP_PROJECT", "")
LOCATION = os.getenv("BQ_LOCATION", "US")

client = bigquery.Client(project=PROJECT, location=LOCATION)  # kênh kết nối

def mb(n):
    # trả None nếu n là None để lỗi đọc quá sớm lộ ra
    return None if n is None else round(n / 1024 / 1024, 2)

def stats(sql, max_mb=100):
    cfg = bigquery.QueryJobConfig(
        use_query_cache=False,
        maximum_bytes_billed=max_mb * 1024 * 1024,  # trần byte bị tính tiền
    )
    job = client.query(sql, job_config=cfg)
    df = job.result().to_dataframe()            # chờ xong rồi lấy kết quả
    print(f"processed: {mb(job.total_bytes_processed)} MB | billed: {mb(job.total_bytes_billed)} MB")
    return df

print("project:", PROJECT, "| location:", LOCATION)
print("thư mục tồn tại:", OLIST_DIR.exists(), "| số file csv:", len(list(OLIST_DIR.glob("*.csv"))))

project: bq-learning-510104 | location: US
thư mục tồn tại: True | số file csv: 9


In [ ]:
# Cell 2
# Mục đích: tạo dataset olist_raw ở US để chứa dữ liệu thô.
ds = bigquery.Dataset(f"{PROJECT}.olist_raw")
ds.location = LOCATION                          # phải khớp vùng, BigQuery không nối chéo vùng
ds = client.create_dataset(ds, exists_ok=True)  # chạy lại không báo lỗi
print("dataset:", ds.dataset_id, "| location:", ds.location)

dataset: olist_raw | location: US


In [8]:
# Cell 3
# Mục đích: nạp translation (71 dòng) vào olist_raw, mọi cột là STRING.
# Tương đương SQL: CREATE OR REPLACE TABLE ... với 2 cột STRING rồi nạp file vào.
def load_csv_as_string(csv_path, table_name):
    # đọc dòng tiêu đề để lấy tên cột (nrows=0 chỉ đọc tiêu đề, giống Ngày 1)
    import pandas as pd
    cols = pd.read_csv(csv_path, nrows=0).columns.tolist()
    # schema = [bigquery.SchemaField(c, "STRING") for c in cols]   # khai báo rõ, tất cả STRING
    schema = []
    for c in cols:
        schema.append(bigquery.SchemaField(c,"STRING"))

    cfg = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.CSV,
        skip_leading_rows=1,                         # bỏ dòng tiêu đề
        schema=schema,
        write_disposition="WRITE_TRUNCATE",          # chạy lại không bị trùng (idempotent)
        allow_quoted_newlines=True,                  # cho phép xuống dòng trong ô (dùng cho reviews)
    )
    table_id = f"{PROJECT}.olist_raw.{table_name}"
    with open(csv_path, "rb") as f:
        job = client.load_table_from_file(f, table_id, job_config=cfg)
    job.result()                                     # chờ load job xong
    print(f"{table_name}: gửi lên {job.output_rows} dòng, {len(cols)} cột")
    return job

csv_path = OLIST_DIR / "product_category_name_translation.csv"
job = load_csv_as_string(csv_path, "product_category_translation")

product_category_translation: gửi lên 71 dòng, 2 cột


In [ ]:
# Cell 4
# Mục đích: đối chiếu độc lập bằng SQL, và xem kiểu cột thật trên BigQuery.
t = client.get_table(f"{PROJECT}.olist_raw.product_category_translation")
print("số dòng:", t.num_rows)
for field in t.schema:
    print(field.name, field.field_type)

sql = f"""
SELECT COUNT(*) AS so_dong,
       COUNT(DISTINCT product_category_name) AS so_category
FROM `{PROJECT}.olist_raw.product_category_translation`
"""
df = stats(sql)
print(len(df))
df

số dòng: 71
product_category_name STRING
product_category_name_english STRING
processed: 0.0 MB | billed: 10.0 MB
1


,so_dong,so_category
0,71,71
